# Dagua Tutorial Walkthrough

This notebook is the main hands-on launching point for Dagua.

It is organized in three layers:
1. Easy: get a graph on the page fast
2. Medium: styling, IO, routing, export, and cinematic outputs
3. Advanced: how to fix a particular visual issue in place

The goal is not to enumerate every option. The goal is to touch the full surface area in paradigmatic form so you can confidently go build your own thing, then use the glossary for precise reference.

In [ ]:
from pathlib import Path
import torch
import dagua
from dagua import (
    DaguaGraph,
    LayoutConfig,
    NodeStyle,
    EdgeStyle,
    ClusterStyle,
    Flex,
    LayoutFlex,
    AlignGroup,
    AnimationConfig,
    PosterConfig,
    TourConfig,
)

OUTDIR = Path('/tmp/dagua-tutorial')
OUTDIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTDIR

## Easy: Get a Graph on the Page

Start with the highest-leverage entrypoints: `from_edge_list`, `layout`, and `draw`.

In [ ]:
g = DaguaGraph.from_edge_list([
    ('raw_data', 'clean'),
    ('clean', 'features'),
    ('features', 'model'),
    ('model', 'score'),
    ('score', 'decision'),
])
g

In [ ]:
fig, ax = dagua.draw(g, LayoutConfig(device=DEVICE, steps=70, edge_opt_steps=8, seed=42))
fig

If you want more control, separate layout from rendering.

In [ ]:
config = LayoutConfig(device=DEVICE, steps=70, edge_opt_steps=8, seed=42)
positions = dagua.layout(g, config)
positions[:3]

## Medium: Use the Broader Surface Area

This section touches graph construction, styles, clusters, routing, IO, and export.

In [ ]:
g = DaguaGraph(direction='TB')
g.add_node('input', label='User Request', type='input', style=NodeStyle(shape='ellipse'))
g.add_node('router', label='Router')
g.add_node('search', label='Search Tool', style=NodeStyle(base_color='#0072B2'))
g.add_node('reason', label='Reasoning Core')
g.add_node('safety', label='Safety Review', style=NodeStyle(base_color='#E69F00'))
g.add_node('output', label='Final Answer', type='output', style=NodeStyle(base_color='#009E73'))

g.add_edge('input', 'router', label='intent')
g.add_edge('router', 'search', label='tool call', style=EdgeStyle(routing='ortho'))
g.add_edge('router', 'reason', label='context', style=EdgeStyle(routing='straight', curvature=0.0))
g.add_edge('search', 'reason', label='retrieved docs')
g.add_edge('reason', 'safety', label='draft')
g.add_edge('safety', 'output', label='approved')

g.add_cluster('Core System', ['router', 'search', 'reason', 'safety'], label='Core System', style=ClusterStyle())
g.compute_node_sizes()
fig, ax = dagua.draw(g, LayoutConfig(device=DEVICE, steps=90, edge_opt_steps=10, seed=42))
fig

In [ ]:
config = LayoutConfig(device=DEVICE, steps=80, edge_opt_steps=10, seed=42)
positions = dagua.layout(g, config)
g.compute_node_sizes()
curves = dagua.route_edges(positions, g.edge_index, g.node_sizes, g.direction, g)
label_positions = dagua.place_edge_labels(curves, positions, g.node_sizes, g.edge_labels, g)
fig, ax = dagua.render(g, positions, config, curves=curves, label_positions=label_positions)
fig

In [ ]:
json_path = OUTDIR / 'assistant_graph.json'
yaml_path = OUTDIR / 'assistant_graph.yaml'
dagua.save(g, json_path)
dagua.save(g, yaml_path)
g_json = dagua.load(json_path)
g_yaml = dagua.load(yaml_path)
{'json_nodes': g_json.num_nodes, 'yaml_nodes': g_yaml.num_nodes, 'json_path': str(json_path), 'yaml_path': str(yaml_path)}

### Image to graph, code, and theme

This is the most magical IO surface. The same image can become a live `DaguaGraph`, a raw dict, or editable best-practice Dagua code.

Canonical setup is to configure the provider once, ideally using an env var name rather than hardcoding a secret in the notebook.

In [ ]:
dagua.configure_image_ai(provider='openai', api_key_env='OPENAI_API_KEY')

# Then use whichever return surface matches your goal:
# graph = dagua.from_image('diagram.png')
# graph_dict = dagua.graph_dict_from_image('diagram.png')
# graph_code = dagua.graph_code_from_image('diagram.png')
# graph_script = dagua.graph_code_from_image('diagram.png', include_demo_script=True)
# theme = dagua.theme_from_image('reference.png')
# theme_dict = dagua.theme_dict_from_image('reference.png')
# theme_code = dagua.theme_code_from_image('reference.png')

dagua.get_image_ai_config()

In [ ]:
png_path = OUTDIR / 'assistant_graph.png'
svg_path = OUTDIR / 'assistant_graph.svg'
pdf_path = OUTDIR / 'assistant_graph.pdf'

dagua.draw(g, LayoutConfig(device=DEVICE, steps=60, edge_opt_steps=8, seed=42), output=str(png_path))
dagua.draw(g, LayoutConfig(device=DEVICE, steps=60, edge_opt_steps=8, seed=42), output=str(svg_path))
dagua.draw(g, LayoutConfig(device=DEVICE, steps=60, edge_opt_steps=8, seed=42), output=str(pdf_path))
[str(png_path), str(svg_path), str(pdf_path)]

In [ ]:
positions = dagua.layout(g, LayoutConfig(device=DEVICE, steps=60, edge_opt_steps=8, seed=42))
poster_path = OUTDIR / 'assistant_poster.png'
tour_path = OUTDIR / 'assistant_tour.gif'
optimization_path = OUTDIR / 'assistant_optimization.gif'

poster_result = dagua.poster(g, positions=positions, output=str(poster_path), poster_config=PosterConfig(scene='auto'))
tour_result = dagua.tour(g, positions=positions, output=str(tour_path), tour_config=TourConfig(scene='zoom_pan', fps=18))
optimization_result = dagua.animate(g, LayoutConfig(device=DEVICE, steps=45, edge_opt_steps=8, seed=42), output=str(optimization_path), animation_config=AnimationConfig(fps=16, dpi=110, max_layout_frames=18, max_edge_frames=8))

{'poster': poster_result.output, 'tour': tour_result.output, 'optimization': optimization_result.output}

## Advanced: Fix a Particular Visual Problem in Place

A useful mental model is:
- If the graph needs more room, change spacing (`node_sep`, `rank_sep`)
- If a node must stay somewhere, pin it
- If peers should line up, use alignment
- If the edge style is the issue, change routing/curvature rather than the whole layout
- If a region should read as a unit, use clusters and cluster labels

### Example 1: The graph feels cramped

In [ ]:
g_cramped = DaguaGraph.from_edge_list([
    ('in', 'a'), ('in', 'b'), ('in', 'c'),
    ('a', 'merge'), ('b', 'merge'), ('c', 'merge'),
    ('merge', 'out'),
])

tight = LayoutConfig(device=DEVICE, steps=60, edge_opt_steps=6, node_sep=14, rank_sep=28, seed=42)
wide = LayoutConfig(device=DEVICE, steps=60, edge_opt_steps=6, node_sep=46, rank_sep=80, seed=42)

fig_tight, _ = dagua.draw(g_cramped, tight)
fig_wide, _ = dagua.draw(g_cramped, wide)
fig_wide

### Example 2: A node or output must stay in a known place

In [ ]:
g_pin = DaguaGraph.from_edge_list([
    ('input', 'prep'), ('prep', 'branch_a'), ('prep', 'branch_b'),
    ('branch_a', 'merge'), ('branch_b', 'merge'), ('merge', 'output'),
])
g_pin.compute_node_sizes()
g_pin.flex = LayoutFlex(
    pins={
        'input': (Flex.locked(0.0), Flex.locked(0.0)),
        'output': (Flex.locked(260.0), Flex.locked(120.0)),
    }
)
fig, ax = dagua.draw(g_pin, LayoutConfig(device=DEVICE, steps=80, edge_opt_steps=8, seed=42))
fig

The optimization animation is especially good here because it makes the constraint visible: the pinned nodes stay anchored while the rest of the graph settles around them.

In [ ]:
pin_animation_path = OUTDIR / 'pinned_layout.gif'
pin_animation = dagua.animate(
    g_pin,
    LayoutConfig(device=DEVICE, steps=55, edge_opt_steps=8, seed=42),
    output=str(pin_animation_path),
    animation_config=AnimationConfig(
        fps=16,
        dpi=110,
        max_layout_frames=20,
        max_edge_frames=8,
        camera='focus',
        center_on='merge',
    ),
)
pin_animation

### Example 3: Parallel peers should align cleanly

In [ ]:
g_align = DaguaGraph.from_edge_list([
    ('input', 'vision'), ('input', 'text'), ('input', 'tooling'),
    ('vision', 'merge'), ('text', 'merge'), ('tooling', 'merge'), ('merge', 'out'),
])
g_align.compute_node_sizes()
g_align.flex = LayoutFlex(align_y=[AlignGroup(['vision', 'text', 'tooling'], weight=6.0)])
fig, ax = dagua.draw(g_align, LayoutConfig(device=DEVICE, steps=80, edge_opt_steps=8, seed=42))
fig

Animation also helps explain alignment constraints. Watching the three peers snap into a clean row makes the purpose of `AlignGroup` much clearer than a static before/after.

In [ ]:
align_animation_path = OUTDIR / 'aligned_layout.gif'
align_animation = dagua.animate(
    g_align,
    LayoutConfig(device=DEVICE, steps=55, edge_opt_steps=8, seed=42),
    output=str(align_animation_path),
    animation_config=AnimationConfig(
        fps=16,
        dpi=110,
        max_layout_frames=20,
        max_edge_frames=8,
        camera='global',
    ),
)
align_animation

### Example 4: The edges are the visual problem, not the node layout

In [ ]:
g_edges = DaguaGraph.from_edge_list([
    ('input', 'fork'), ('fork', 'a'), ('fork', 'b'), ('a', 'merge'), ('b', 'merge'), ('merge', 'out'),
])
g_edges.edge_styles[1] = EdgeStyle(routing='straight', curvature=0.0)
g_edges.edge_styles[2] = EdgeStyle(routing='ortho', curvature=0.0)
fig, ax = dagua.draw(g_edges, LayoutConfig(device=DEVICE, steps=70, edge_opt_steps=8, seed=42))
fig

### Example 5: A subsystem needs to read as a single unit

In [ ]:
g_cluster = DaguaGraph.from_edge_list([
    ('request', 'router'), ('router', 'search'), ('router', 'reason'),
    ('search', 'reason'), ('reason', 'safety'), ('safety', 'answer'),
])
g_cluster.add_cluster('Core', ['router', 'search', 'reason', 'safety'], label='Core System', style=ClusterStyle())
fig, ax = dagua.draw(g_cluster, LayoutConfig(device=DEVICE, steps=80, edge_opt_steps=8, seed=42))
fig

## What Next?

From here, the usual path is:
- keep this notebook open while you prototype
- use the glossary for exact option names and defaults
- use the developer playground notebook when you want broad regression-style UI checks
- add your own graph builders for your domain and iterate with `layout`, `draw`, `poster`, `tour`, and `animate`